# Chapter 2: Summarizing Data
**Artificial Intelligence Engineering | Statistics**

This notebook contains Python implementations of all theoretical concepts from the lecture slides.

---
## Contents
1. [Examining Numerical Data](#1)
   - Scatter Plots
   - Dot Plots
   - Histograms & Bin Width
   - Distribution Shape (Modality & Skewness)
2. [Measures of Center and Spread](#2)
   - Mean & Variance & Standard Deviation
   - Median, Q1, Q3, IQR
   - Box Plot
   - Robust Statistics
3. [Logarithmic Transformation](#3)
4. [Examining Categorical Data](#4)
   - Contingency Tables
   - Bar Charts
   - Mosaic Plots
5. [Case Study: Gender Discrimination & Hypothesis Testing](#5)

In [ ]:
# Required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Color palette
BLUE   = '#4C88C4'
LBLUE  = '#A8C8E8'
ORANGE = '#E8834C'
GREEN  = '#5BAD72'
GRAY   = '#888888'

np.random.seed(42)
print('Libraries loaded successfully ✓')

---
<a id='1'></a>
## 1. Examining Numerical Data

### 1.1 Scatter Plot

> **Definition:** Scatter plots are used to visualize the **relationship between two numerical variables**.

The example below simulates the **life expectancy – total fertility** relationship from the lecture slides.

In [ ]:
# Gapminder-like data simulation
n = 150
# Fertility rate: between 1.1 and 7.5
fertility = np.random.uniform(1.1, 7.5, n)
# Life expectancy: negative linear relationship with fertility + noise
life_exp  = 85 - 6.5 * fertility + np.random.normal(0, 4, n)
life_exp  = np.clip(life_exp, 30, 85)
# Population (for bubble size)
population = np.random.exponential(50, n) + 5

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, alpha, title in zip(axes, [0.6, 0.4], ['2000', '2020']):
    scatter = ax.scatter(
        fertility, life_exp,
        s=population * 3,
        c=np.random.choice([BLUE, ORANGE, GREEN, '#C45CA0'], n),
        alpha=alpha, edgecolors='white', linewidths=0.5
    )
    # Trend line
    z = np.polyfit(fertility, life_exp, 1)
    p = np.poly1d(z)
    x_line = np.linspace(fertility.min(), fertility.max(), 100)
    ax.plot(x_line, p(x_line), color=ORANGE, linewidth=2, linestyle='--', alpha=0.8, label='Trend')
    ax.set_xlabel('Children per Woman (Total Fertility Rate)', fontsize=10)
    ax.set_ylabel('Life Expectancy (years)', fontsize=10)
    ax.set_title(f'Life Expectancy vs Fertility Rate — {title}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Scatter Plot Example (Gapminder-like)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Correlation coefficient
r, p_val = stats.pearsonr(fertility, life_exp)
print(f'Pearson Correlation Coefficient (r) = {r:.3f}')
print(f'Interpretation: As fertility increases, life expectancy {"decreases" if r < 0 else "increases"} '
      f'(negative linear relationship).')

### 1.2 Dot Plot and Mean

> **Definition:** Used to visualize a single numerical variable. Darker regions represent areas with more observations.

**Sample mean:**
$$\bar{x} = \frac{x_1 + x_2 + \cdots + x_n}{n}$$

In [ ]:
# GPA data simulation
# Mimicking the distribution in the slides: slightly left-skewed, dense between 3.3 and 4.0
gpa_data = np.concatenate([
    np.random.normal(3.6, 0.3, 200),
    np.random.uniform(2.5, 3.0, 30)
])
gpa_data = np.clip(gpa_data, 2.5, 4.0)

mean_gpa   = np.mean(gpa_data)
median_gpa = np.median(gpa_data)

fig, axes = plt.subplots(2, 1, figsize=(10, 6))

# --- Dot plot (with jitter) ---
ax = axes[0]
y_jitter = np.random.uniform(-0.15, 0.15, len(gpa_data))
ax.scatter(gpa_data, y_jitter, alpha=0.35, s=18, color=BLUE, edgecolors='none')
ax.axvline(mean_gpa, color=ORANGE, linewidth=2, label=f'Mean = {mean_gpa:.2f}')
ax.plot(mean_gpa, 0, marker='^', color=ORANGE, markersize=14, zorder=5)
ax.set_yticks([])
ax.set_xlabel('GPA', fontsize=11)
ax.set_xlim(2.4, 4.05)
ax.set_title('Dot Plot — GPA Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# --- Stacked dot plot (histogram style) ---
ax2 = axes[1]
ax2.hist(gpa_data, bins=30, color=BLUE, alpha=0.7, edgecolor='white')
ax2.axvline(mean_gpa,   color=ORANGE, linewidth=2, linestyle='-',  label=f'Mean   = {mean_gpa:.2f}')
ax2.axvline(median_gpa, color=GREEN,  linewidth=2, linestyle='--', label=f'Median = {median_gpa:.2f}')
ax2.set_xlabel('GPA', fontsize=11)
ax2.set_ylabel('Frequency', fontsize=11)
ax2.set_title('Stacked Dot Plot (Histogram) — GPA Distribution', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f'n                      = {len(gpa_data)}')
print(f'Sample Mean (x̄)        = {mean_gpa:.3f}')
print(f'Median                 = {median_gpa:.3f}')

### 1.3 Histograms and Bin Width

> **Important:** The chosen **bin width** can change the story a histogram tells.

Too wide bins → too much information is hidden  
Too narrow bins → too much noise is visible

In [ ]:
# Extracurricular activities data (right-skewed)
extracurricular = np.concatenate([
    np.random.exponential(8, 150),
    np.random.uniform(0, 5, 50),
    [60, 65, 70]  # outliers
])
extracurricular = np.clip(extracurricular, 0, 75)

bin_configs = [
    (5,  'Too Wide\n(too much is hidden)'),
    (10, 'Good Bin Width\n(balanced)'),
    (20, 'Moderate Bin Width'),
    (50, 'Too Narrow\n(too much detail / noise)'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, (bins, title) in zip(axes, bin_configs):
    ax.hist(extracurricular, bins=bins, color=BLUE, alpha=0.75, edgecolor='white')
    ax.set_xlabel('Hours per Week (Extracurricular Activities)', fontsize=9)
    ax.set_ylabel('Frequency', fontsize=9)
    ax.set_title(f'{title}  (bins={bins})', fontsize=10, fontweight='bold')

plt.suptitle('Effect of Bin Width on Histograms', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.4 Distribution Shape: Modality and Skewness

| Feature | Terms |
|---------|----------|
| Modality | Unimodal, Bimodal, Multimodal, Uniform |
| Skewness | Right-skewed, Left-skewed, Symmetric |

> **Rule:** Histograms are said to be skewed toward the side with the **longer tail**.

In [ ]:
# 6 different distribution shapes
dists = {
    'Unimodal\n(Symmetric)':  np.random.normal(10, 2, 1000),
    'Bimodal':                np.concatenate([np.random.normal(5, 1, 500),
                                               np.random.normal(15, 1, 500)]),
    'Multimodal':             np.concatenate([np.random.normal(3, 0.8, 300),
                                               np.random.normal(8, 0.8, 300),
                                               np.random.normal(14, 0.8, 400)]),
    'Uniform Distribution':   np.random.uniform(0, 20, 1000),
    'Right-Skewed':           np.random.exponential(3, 1000),
    'Left-Skewed':            20 - np.random.exponential(3, 1000),
}

colors = [BLUE, GREEN, ORANGE, '#9B59B6', '#E74C3C', '#1ABC9C']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, (title, data), color in zip(axes, dists.items(), colors):
    ax.hist(data, bins=35, color=color, alpha=0.8, edgecolor='white')
    mean_val   = np.mean(data)
    median_val = np.median(data)
    ax.axvline(mean_val,   color='navy',  linewidth=2, linestyle='--', label=f'Mean = {mean_val:.1f}')
    ax.axvline(median_val, color='black', linewidth=2, linestyle='-',  label=f'Med. = {median_val:.1f}')
    skewness = stats.skew(data)
    ax.set_title(f'{title}\n(skewness = {skewness:.2f})', fontsize=10, fontweight='bold')
    ax.set_yticks([])
    ax.legend(fontsize=8)

plt.suptitle('Distribution Shapes: Modality and Skewness', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Skewness Interpretation:')
print('  skewness > 0  → Right-skewed  (mean > median)')
print('  skewness < 0  → Left-skewed   (mean < median)')
print('  skewness ≈ 0  → Symmetric     (mean ≈ median)')

---
<a id='2'></a>
## 2. Measures of Center and Spread

### 2.1 Variance and Standard Deviation

**Variance** — the average squared deviation from the mean:
$$s^2 = \frac{\sum_{i=1}^{n}(x_i - \bar{x})^2}{n-1}$$

**Standard Deviation** — the square root of variance (same units as the data):
$$s = \sqrt{s^2}$$

> **Why $n-1$?** Loss of degrees of freedom (Bessel's correction) — ensures the sample statistic is an unbiased estimator of the population variance.

In [ ]:
# Sleep data from the slides
sleep_data = np.concatenate([
    np.random.normal(7, 1.5, 180),
    np.random.normal(4, 0.5, 20),
    np.random.normal(10, 0.5, 17)
])
sleep_data = np.clip(sleep_data, 2, 12)

# --- Manual calculation ---
n      = len(sleep_data)
x_bar  = np.mean(sleep_data)
var_s2 = np.sum((sleep_data - x_bar)**2) / (n - 1)   # sample variance
std_s  = np.sqrt(var_s2)

# --- Verification with NumPy ---
assert abs(var_s2 - np.var(sleep_data, ddof=1)) < 1e-10
assert abs(std_s  - np.std(sleep_data, ddof=1)) < 1e-10

# --- Visualization ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(sleep_data, bins=20, color=BLUE, alpha=0.75, edgecolor='white', label='Observations')
ax.axvline(x_bar, color=ORANGE, linewidth=2.5, label=f'Mean (x̄) = {x_bar:.2f} hours')

# ±1s, ±2s, and ±3s bands
for k, alpha in [(1, 0.20), (2, 0.12), (3, 0.07)]:
    ax.axvspan(x_bar - k*std_s, x_bar + k*std_s, alpha=alpha, color=ORANGE,
               label=f'±{k}s  ({x_bar - k*std_s:.1f} – {x_bar + k*std_s:.1f})')

ax.set_xlabel('Nightly Sleep Duration (hours)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Sleep Duration: Mean and Standard Deviation', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

print('=' * 45)
print(f'Sample Size        n  = {n}')
print(f'Sample Mean        x̄  = {x_bar:.2f} hours')
print(f'Sample Variance    s² = {var_s2:.2f} hours²')
print(f'Standard Deviation s  = {std_s:.2f} hours')
print('=' * 45)

# How many data points fall within 3s?
within_3s = np.sum(np.abs(sleep_data - x_bar) <= 3 * std_s)
print(f'\nData within 3 standard deviations: {within_3s}/{n} = {within_3s/n*100:.1f}%')

### 2.2 Median, Quartiles, and IQR

| Concept | Definition |
|--------|-------|
| **Median** | The value that splits the data in half — 50th percentile |
| **Q1** | 25th percentile (first quartile) |
| **Q3** | 75th percentile (third quartile) |
| **IQR** | Q3 − Q1 (middle 50% range) |

In [ ]:
# Small example — step-by-step calculation
sample = [4, 7, 13, 2, 1, 9, 6, 11, 5]
sample_sorted = sorted(sample)
print('Sorted data:', sample_sorted)

median_val = np.median(sample)
q1         = np.percentile(sample, 25)
q3         = np.percentile(sample, 75)
iqr        = q3 - q1

print(f'Median           = {median_val}')
print(f'Q1 (25th pctile) = {q1}')
print(f'Q3 (75th pctile) = {q3}')
print(f'IQR              = Q3 - Q1 = {q3} - {q1} = {iqr}')

# Even-numbered sample
sample2 = [0, 1, 2, 3, 4, 5]
print(f'\nEven-numbered sample {sample2}: Median = ({sample2[2]} + {sample2[3]}) / 2 = {np.median(sample2)}')

### 2.3 Box Plot

> **Whisker Rule:** Whiskers extend up to **1.5 × IQR** beyond the quartiles.
> - Upper whisker fence = Q3 + 1.5 × IQR
> - Lower whisker fence = Q1 − 1.5 × IQR
> - Points beyond these fences are flagged as **potential outliers**.

In [ ]:
# Weekly study hours data (right-skewed + outliers)
study_hours = np.concatenate([
    np.random.normal(15, 6, 180),
    [45, 50, 55, 60, 65, 70]  # outliers
])
study_hours = np.clip(study_hours, 0, 75)

Q1           = np.percentile(study_hours, 25)
Q3           = np.percentile(study_hours, 75)
IQR          = Q3 - Q1
lower_fence  = Q1 - 1.5 * IQR
upper_fence  = Q3 + 1.5 * IQR

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# --- Annotated box plot ---
ax = axes[0]
bp = ax.boxplot(study_hours, vert=True, patch_artist=True,
                boxprops=dict(facecolor=LBLUE, color=BLUE, linewidth=2),
                medianprops=dict(color='black', linewidth=2.5),
                whiskerprops=dict(color=BLUE, linewidth=1.5),
                capprops=dict(color=BLUE, linewidth=2),
                flierprops=dict(marker='o', color=ORANGE, alpha=0.7, markersize=6))

median_val = np.median(study_hours)

# Annotations
offset = 0.55
for label, yval, color in [
    (f'Q1 = {Q1:.1f}',                           Q1,                      GREEN),
    (f'Median = {median_val:.1f}',               median_val,              'black'),
    (f'Q3 = {Q3:.1f}',                           Q3,                      ORANGE),
    (f'Lower Whisker ≈ {max(lower_fence,0):.1f}',max(lower_fence, 0),     BLUE),
    (f'Upper Whisker ≈ {upper_fence:.1f}',       upper_fence,             BLUE),
]:
    ax.annotate(label, xy=(1, yval), xytext=(1 + offset, yval),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2),
                fontsize=9, color=color, va='center')

ax.set_ylabel('Weekly Study Hours', fontsize=11)
ax.set_title('Anatomy of a Box Plot', fontsize=12, fontweight='bold')
ax.set_xticks([])
ax.set_xlim(0.5, 2.2)

# --- Box plot + data points ---
ax2 = axes[1]
jitter = np.random.uniform(-0.12, 0.12, len(study_hours))
outliers_mask = (study_hours > upper_fence) | (study_hours < lower_fence)
ax2.scatter(1 + jitter[~outliers_mask], study_hours[~outliers_mask],
            alpha=0.3, s=15, color=BLUE, label='Normal observations')
ax2.scatter(1 + jitter[outliers_mask], study_hours[outliers_mask],
            alpha=0.8, s=40, color=ORANGE, label='Potential outlier', zorder=5)
ax2.boxplot(study_hours, vert=True, patch_artist=True,
            boxprops=dict(facecolor=LBLUE, color=BLUE, alpha=0.4, linewidth=1.5),
            medianprops=dict(color='black', linewidth=2),
            whiskerprops=dict(color=BLUE, linewidth=1.5),
            capprops=dict(color=BLUE, linewidth=2),
            showfliers=False)
ax2.set_ylabel('Weekly Study Hours', fontsize=11)
ax2.set_title('Box Plot + Data Points', fontsize=12, fontweight='bold')
ax2.set_xticks([])
ax2.legend(fontsize=9)

plt.suptitle('Box Plot', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Q1  = {Q1:.2f}    Q3  = {Q3:.2f}    IQR = {IQR:.2f}')
print(f'Lower whisker fence = Q1 - 1.5×IQR = {lower_fence:.2f}')
print(f'Upper whisker fence = Q3 + 1.5×IQR = {upper_fence:.2f}')
print(f'Number of outliers  = {outliers_mask.sum()}')

### 2.4 Robust Statistics

| Statistic | Resistant to Outliers | Recommended Use |
|------------|---------------------|--------------------|
| **Median & IQR** | ✅ Robust | Skewed distributions |
| **Mean & s** | ❌ Sensitive | Symmetric distributions |

In [ ]:
# Household income data (reproducing the table from the slides)
np.random.seed(7)
income = np.concatenate([
    np.random.lognormal(np.log(190_000), 0.5, 200),
    np.random.uniform(800_000, 1_000_000, 20)
])

def summary_table(data, label):
    Q1  = np.percentile(data, 25)
    Q3  = np.percentile(data, 75)
    return {
        'Scenario':   label,
        'Median':     f'{np.median(data):>12,.0f}',
        'IQR':        f'{Q3-Q1:>12,.0f}',
        'Mean':       f'{np.mean(data):>12,.0f}',
        'Std.Dev':    f'{np.std(data, ddof=1):>12,.0f}',
    }

# 3 scenarios
income_max_changed = income.copy(); income_max_changed[np.argmax(income)] = 10_000_000
income_min_changed = income.copy(); income_min_changed[np.argmin(income)] = 10_000_000

df_summary = pd.DataFrame([
    summary_table(income,             'Original data'),
    summary_table(income_max_changed, 'Largest → 10 million'),
    summary_table(income_min_changed, 'Smallest → 10 million'),
])
df_summary.set_index('Scenario', inplace=True)

print('Robust Statistics Comparison')
print('=' * 72)
print(df_summary.to_string())
print('=' * 72)
print('\n→ Median and IQR are almost unaffected by outliers (ROBUST)')
print('→ Mean and standard deviation change dramatically (SENSITIVE)')

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
titles   = ['Original', 'Largest → 10M', 'Smallest → 10M']
datasets = [income, income_max_changed, income_min_changed]

for ax, data, title in zip(axes, datasets, titles):
    plot_data = data[data < 2_000_000]  # clip for visualization
    ax.hist(plot_data, bins=25, color=BLUE, alpha=0.75, edgecolor='white')
    ax.axvline(np.median(data), color=GREEN,  linewidth=2, linestyle='-',  label=f'Med={np.median(data)/1e3:.0f}K')
    ax.axvline(np.mean(data),   color=ORANGE, linewidth=2, linestyle='--', label=f'Mean={np.mean(data)/1e3:.0f}K')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Annual Household Income', fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Robust Statistics: Effect of an Outlier', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.5 Side-by-Side Box Plots — Comparing Groups

In [ ]:
# Number of club memberships by year of study
np.random.seed(10)
year_groups = {
    'Freshman':   np.random.poisson(2.5, 80),
    'Sophomore':  np.random.poisson(3.0, 80),
    'Junior':     np.random.poisson(3.2, 80),
    'Senior':     np.random.poisson(2.8, 80),
}

fig, ax = plt.subplots(figsize=(9, 5))
data_list = list(year_groups.values())
labels    = list(year_groups.keys())

bp = ax.boxplot(data_list, labels=labels, patch_artist=True,
                boxprops=dict(facecolor=LBLUE, color=BLUE, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2.5),
                whiskerprops=dict(color=BLUE),
                flierprops=dict(marker='o', color=ORANGE, alpha=0.6, markersize=6))

# Mean markers
for i, data in enumerate(data_list, 1):
    ax.plot(i, np.mean(data), marker='D', color=ORANGE, markersize=8, zorder=5,
            label='Mean' if i == 1 else '')

ax.set_xlabel('Year of Study', fontsize=11)
ax.set_ylabel('Number of Club Memberships', fontsize=11)
ax.set_title('Distribution of Club Memberships by Year of Study\n(Side-by-Side Box Plots)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('Group Summaries:')
for label, data in year_groups.items():
    print(f'  {label:12s}: Mean={np.mean(data):.2f}  Med={np.median(data):.1f}  IQR={np.percentile(data,75)-np.percentile(data,25):.1f}')

---
<a id='3'></a>
## 3. Logarithmic Transformation

> When data is **highly skewed**, a logarithmic transformation can facilitate modeling:
> - Outliers become less pronounced
> - The distribution becomes more symmetric
>
> **Disadvantage:** Interpreting results in log units becomes more difficult.

In [ ]:
# Basketball game attendance data (right-skewed)
np.random.seed(5)
game_count = np.random.exponential(8, 200)
game_count = np.clip(game_count, 0.5, 75)  # minimum 0.5 — log(0) undefined

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Raw data
ax1 = axes[0]
ax1.hist(game_count, bins=25, color=BLUE, alpha=0.75, edgecolor='white')
ax1.axvline(np.mean(game_count),   color=ORANGE, lw=2, linestyle='--', label=f'Mean={np.mean(game_count):.1f}')
ax1.axvline(np.median(game_count), color=GREEN,  lw=2, linestyle='-',  label=f'Med={np.median(game_count):.1f}')
ax1.set_xlabel('Number of Games Attended', fontsize=11)
ax1.set_ylabel('Frequency', fontsize=11)
ax1.set_title(f'Raw Data  (skewness = {stats.skew(game_count):.2f})', fontsize=11, fontweight='bold')
ax1.legend(fontsize=9)

# Log transformation
log_games = np.log(game_count)
ax2 = axes[1]
ax2.hist(log_games, bins=25, color=GREEN, alpha=0.75, edgecolor='white')
ax2.axvline(np.mean(log_games),   color=ORANGE, lw=2, linestyle='--', label=f'Mean={np.mean(log_games):.2f}')
ax2.axvline(np.median(log_games), color='navy', lw=2, linestyle='-',  label=f'Med={np.median(log_games):.2f}')
ax2.set_xlabel('log(Number of Games Attended)', fontsize=11)
ax2.set_ylabel('Frequency', fontsize=11)
ax2.set_title(f'Log Transformation  (skewness = {stats.skew(log_games):.2f})', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle('Logarithmic Transformation: Reducing Skewness', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Conversion table
conversion_table = pd.DataFrame({
    'Game Count':      [70, 50, 25, 10, 1],
    'log(Game Count)': [round(np.log(x), 2) for x in [70, 50, 25, 10, 1]]
})
print('Transformation Table:')
print(conversion_table.to_string(index=False))

---
<a id='4'></a>
## 4. Examining Categorical Data

### 4.1 Contingency Tables

> **Contingency table:** A table that summarizes two categorical variables jointly.  
> Based on the **Titanic** dataset from the slides.

In [ ]:
# Reconstruct Titanic data
titanic_df = pd.DataFrame({
    'Age Group': ['Adult'] * 2092 + ['Child'] * 109,
    'Survived':  (['Died'] * 1438 + ['Survived'] * 654) +
                 (['Died'] * 52   + ['Survived'] * 57)
})

# Contingency table — frequencies
cross_freq = pd.crosstab(
    titanic_df['Age Group'], titanic_df['Survived'],
    margins=True, margins_name='Total'
)
print('Contingency Table — Frequencies')
print(cross_freq)

# Contingency table — row proportions (conditional probability)
cross_row = pd.crosstab(
    titanic_df['Age Group'], titanic_df['Survived'],
    normalize='index'
).round(3)
print('\nContingency Table — Row Proportions (Conditional Probability)')
print(cross_row)

pct_adult = 654 / 2092 * 100
pct_child = 57  / 109  * 100
print(f'\nAdult survival rate: {pct_adult:.1f}%')
print(f'Child survival rate: {pct_child:.1f}%')
print(f'\n→ Children survived at a rate {pct_child - pct_adult:.1f} percentage points higher than adults.')

### 4.2 Bar Charts

Three visualization types from the slides: stacked, side-by-side, and standardized.

In [ ]:
# Data
categories = ['Adult', 'Child']
died       = [1438, 52]
survived   = [654, 57]
totals     = [d + s for d, s in zip(died, survived)]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
x = np.arange(len(categories))
w = 0.5

# 1) Stacked bar chart
ax = axes[0]
ax.bar(x, died,     width=w, label='Died',     color=BLUE,  alpha=0.85)
ax.bar(x, survived, width=w, label='Survived', color=LBLUE, alpha=0.85, bottom=died)
ax.set_xticks(x); ax.set_xticklabels(categories, fontsize=11)
ax.set_ylabel('Frequency'); ax.set_title('Stacked Bar Chart', fontweight='bold')
ax.legend(fontsize=9)

# 2) Side-by-side bar chart
ax2 = axes[1]
ax2.bar(x - w/4, died,     width=w/2, label='Died',     color=BLUE,  alpha=0.85)
ax2.bar(x + w/4, survived, width=w/2, label='Survived', color=LBLUE, alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(categories, fontsize=11)
ax2.set_ylabel('Frequency'); ax2.set_title('Side-by-Side Bar Chart', fontweight='bold')
ax2.legend(fontsize=9)

# 3) Standardized (relative frequency)
ax3 = axes[2]
died_prop     = [d / t for d, t in zip(died,     totals)]
survived_prop = [s / t for s, t in zip(survived, totals)]
ax3.bar(x, died_prop,     width=w, label='Died',     color=BLUE,  alpha=0.85)
ax3.bar(x, survived_prop, width=w, label='Survived', color=LBLUE, alpha=0.85, bottom=died_prop)
ax3.axhline(0.5, color='white', linewidth=1.5, linestyle='--')
ax3.set_xticks(x); ax3.set_xticklabels(categories, fontsize=11)
ax3.set_ylabel('Relative Frequency (Proportion)'); ax3.set_title('Standardized Stacked\nBar Chart', fontweight='bold')
ax3.legend(fontsize=9)
ax3.set_ylim(0, 1)

plt.suptitle('Titanic — Age Group and Survival: Three Visualizations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Mosaic Plot

> A **mosaic plot** differs from a standardized stacked bar chart in that it also scales the **column widths** according to group size.

In [ ]:
from matplotlib.patches import Rectangle

total_n = 2092 + 109  # 2201

# Widths: each group's share of the total
width_adult = 2092 / total_n
width_child = 109  / total_n

# Heights: proportions within each group
prop_died_adult     = 1438 / 2092
prop_survived_adult = 654  / 2092
prop_died_child     = 52   / 109
prop_survived_child = 57   / 109

fig, ax = plt.subplots(figsize=(9, 6))
gap       = 0.02
x_adult   = 0
x_child   = width_adult + gap

# Adult — Died
ax.add_patch(Rectangle((x_adult, prop_survived_adult),
                         width_adult, prop_died_adult,
                         color=BLUE, alpha=0.85))
# Adult — Survived
ax.add_patch(Rectangle((x_adult, 0),
                         width_adult, prop_survived_adult,
                         color=LBLUE, alpha=0.85))
# Child — Died
ax.add_patch(Rectangle((x_child, prop_survived_child),
                         width_child, prop_died_child,
                         color=BLUE, alpha=0.85))
# Child — Survived
ax.add_patch(Rectangle((x_child, 0),
                         width_child, prop_survived_child,
                         color=LBLUE, alpha=0.85))

# Boundary lines
for xpos, width, prop_s in [
    (x_adult, width_adult, prop_survived_adult),
    (x_child, width_child, prop_survived_child)
]:
    ax.plot([xpos, xpos + width], [prop_s, prop_s], 'white', linewidth=2.5)

# Labels
ax.text(x_adult + width_adult/2, 0.5,  'Adult', ha='center', va='center',
        fontsize=13, fontweight='bold', color='white')
ax.text(x_child + width_child/2, 0.5,  'Child', ha='center', va='center',
        fontsize=11, fontweight='bold', color='white')
ax.text(-0.06, prop_survived_adult + prop_died_adult/2, 'Died',     ha='right', fontsize=11)
ax.text(-0.06, prop_survived_adult/2,                   'Survived', ha='right', fontsize=11)

# Size annotations
ax.text(x_adult + width_adult/2, -0.07,
        f'n = 2092\n({width_adult*100:.1f}%)', ha='center', fontsize=9, color=GRAY)
ax.text(x_child + width_child/2, -0.07,
        f'n = 109\n({width_child*100:.1f}%)', ha='center', fontsize=9, color=GRAY)

legend_patches = [mpatches.Patch(color=BLUE,  label='Died'),
                  mpatches.Patch(color=LBLUE, label='Survived')]
ax.legend(handles=legend_patches, loc='upper right', fontsize=10)
ax.set_xlim(-0.08, 1.0)
ax.set_ylim(-0.15, 1.05)
ax.set_xlabel('Age Group  (column width = group size)', fontsize=10)
ax.set_ylabel('Proportion', fontsize=10)
ax.set_title('Mosaic Plot — Titanic (Age Group × Survival)', fontsize=12, fontweight='bold')
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_xticks([])
plt.tight_layout()
plt.show()

---
<a id='5'></a>
## 5. Case Study: Gender Discrimination & Hypothesis Testing

**Study:** 48 male bank managers were each given the same personnel file.  
- 24 managers saw the file as belonging to a **male** candidate, 24 saw it as belonging to a **female** candidate.
- 35 files were recommended for promotion.

**Hypotheses:**  
- $H_0$: Promotion and gender are **independent** — any difference is due to chance alone.  
- $H_A$: Promotion and gender are **dependent** — there is gender-based discrimination.

In [ ]:
# Observed data
obs_male_promoted   = 21
obs_female_promoted = 14
n_male = n_female   = 24

obs_diff = obs_male_promoted/n_male - obs_female_promoted/n_female

df_observed = pd.DataFrame({
    'Promoted':     [21, 14, 35],
    'Not Promoted': [ 3, 10, 13],
    'Total':        [24, 24, 48]
}, index=['Male', 'Female', 'Total'])

print('Observed Data:')
print(df_observed)
print(f'\nPromotion rate — Male   : {obs_male_promoted}/{n_male} = {obs_male_promoted/n_male:.3f} ({obs_male_promoted/n_male*100:.1f}%)')
print(f'Promotion rate — Female : {obs_female_promoted}/{n_female} = {obs_female_promoted/n_female:.3f} ({obs_female_promoted/n_female*100:.1f}%)')
print(f'Observed difference     : {obs_diff:.3f} ({obs_diff*100:.1f}%)')

In [ ]:
# Permutation test (software version of the card-shuffling simulation from the slides)
np.random.seed(2024)

n_sim    = 10_000
n_promo  = 35   # total number of promotions is fixed
n_total  = 48

# Each simulation: shuffle 48 files, count first 24 as male and last 24 as female
files = np.array([1]*35 + [0]*13)  # 1=promoted, 0=not promoted

sim_diffs = []
for _ in range(n_sim):
    np.random.shuffle(files)
    male_rate   = files[:24].mean()
    female_rate = files[24:].mean()
    sim_diffs.append(male_rate - female_rate)

sim_diffs = np.array(sim_diffs)

# p-value: probability of obtaining a result at least as extreme as the observed difference
p_value = np.mean(sim_diffs >= obs_diff)

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(sim_diffs, bins=40, color=BLUE, alpha=0.7, edgecolor='white',
        label=f'Simulated distribution (n={n_sim:,})')
ax.axvline(obs_diff, color=ORANGE, linewidth=3,
           label=f'Observed difference = {obs_diff:.3f}')

# Simulations at least as extreme as the observed difference
ax.hist(sim_diffs[sim_diffs >= obs_diff], bins=40,
        color=ORANGE, alpha=0.6, edgecolor='white', label='p-value region')

ax.set_xlabel('Difference in Promotion Rates (Male − Female)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Permutation Test: Null Distribution of Promotion Rate Difference\n'
             '(H₀: Gender and promotion are independent)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# p-value box
ax.text(0.97, 0.92, f'p-value = {p_value:.4f}',
        transform=ax.transAxes, ha='right', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

print('=' * 55)
print(f'Number of simulations      : {n_sim:,}')
print(f'Observed difference        : {obs_diff:.4f}  ({obs_diff*100:.1f}%)')
print(f'Simulated p-value          : {p_value:.4f}')
print('=' * 55)
if p_value < 0.05:
    print('DECISION (α=0.05): REJECT H₀.')
    print('→ There is statistical evidence of gender-based discrimination.')
else:
    print('DECISION (α=0.05): Fail to reject H₀.')
    print('→ The observed difference may be due to chance.')

In [ ]:
# Additional: Chi-square test (theoretical approach)
from scipy.stats import chi2_contingency

contingency = np.array([[21, 3],   # male:   promoted, not promoted
                         [14, 10]]) # female: promoted, not promoted

chi2, p_chi2, dof, expected = chi2_contingency(contingency, correction=False)

print('Chi-Square Test of Independence')
print(f'χ² statistic        = {chi2:.4f}')
print(f'Degrees of freedom  = {dof}')
print(f'p-value             = {p_chi2:.4f}')
print(f'\nExpected frequencies:')
print(pd.DataFrame(expected.round(2),
                   index=['Male', 'Female'],
                   columns=['Promoted', 'Not Promoted']))
if p_chi2 < 0.05:
    print('\n→ p < 0.05: H₀ is rejected. Gender and promotion are dependent.')

---
## Summary: Hypothesis Testing Framework

| Step | Description |
|------|----------|
| 1 | **H₀** (null hypothesis): The claim that "nothing is happening" — the status quo |
| 2 | **Hₐ** (alternative hypothesis): The research question — "something is happening" |
| 3 | Simulate the distribution of the test statistic **assuming H₀ is true** |
| 4 | Compute the **p-value**: probability that the observed result is at least this extreme |
| 5 | If p < α, **reject** H₀; otherwise **retain** H₀ (never say "H₀ is true"!) |

---
*Notebook prepared based on the Chapter 2: Summarizing Data lecture slides.*